# PhyloGriffin v3 -- Training on Google Colab

This notebook trains the complete PhyloGriffin-v3 model for phylogenetic tree inference.

## Prerequisites
- Google Colab with GPU runtime (T4 or A100 recommended)
- Google Drive mounted for checkpoint storage
- PhyloGriffin package uploaded to your Drive or cloned from GitHub

## Training Stages
1. Masked column reconstruction (self-supervised, ~2-4 hours)
2. Contrastive phylogenetic embedding (~3-6 hours)
3. Graph predictor (~1-2 hours)
4. Diffusion denoiser (~4-8 hours)
5. Supertree reconciler (~2-4 hours)
6. Refinement pass (~1 hour)

Total estimated time: 13-25 hours. Save checkpoints frequently.

In [1]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install numpy scipy dendropy tqdm matplotlib

# Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

# Check GPU
!nvidia-smi

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Looking in indexes: https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.1/465.1 kB 17.8 MB/s eta 0:00:00
Mounted at /content/drive
Sat Jul 18 02:18:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB

In [2]:
import os
import sys

# Option A: If the package is in your Google Drive
PACKAGE_PATH = "/content/drive/MyDrive/phylogriffin"
if os.path.exists(PACKAGE_PATH):
    !cp -r {PACKAGE_PATH} /content/phylogriffin
    print("Copied package from Drive")
else:
    # Option B: Clone from GitHub (if uploaded)
    !git clone https://github.com/amichae2/PhyloGriffin_V3.git /content/phylogriffin
    print("Cloned from GitHub")

sys.path.insert(0, "/content")
os.chdir("/content/phylogriffin")
!pip install -e .

# Verify imports
from torch.utils.data import DataLoader

from phylogriffin.config import PhyloGriffinConfig
from phylogriffin.inference import PhyloGriffinV3, infer_tree
from phylogriffin.model.column_processor import ColumnProcessor
from phylogriffin.model.decomposition import HierarchicalDecomposition
from phylogriffin.model.diffusion import DiffusionTreeGenerator
from phylogriffin.model.graph_predictor import GraphPredictor
from phylogriffin.model.refinement import RefinementPass
from phylogriffin.model.supertree import SupertreeReconciler
from phylogriffin.simulation import evolve_sequences, simulate_yule_tree
from phylogriffin.tree_utils import newick_to_splits, robinson_foulds

print("All imports successful!")

Cloning into '/content/phylogriffin'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 198 (delta 128), reused 123 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 149.67 KiB | 7.48 MiB/s, done.
Resolving deltas: 100% (128/128), done.
Cloned from GitHub
Obtaining file:///content/phylogriffin
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.8/177.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.0 MB/s eta 0:00:00
  Building editable for phylogriffin (pyproject.toml) ... done
  Created wheel for phylogriffin: filename=phylogriffin-0.3.0-0.editable-py3-none-any.whl size=3700 sha256=b343cd6ed5cefe565734a833d7af5cddf3931765098

In [3]:
# Initialize configuration
config = PhyloGriffinConfig(
    alphabet_size=21,  # protein
    gap_idx=20,
    pad_idx=21,
)

# Override for smaller model (Colab-friendly)
config.griffin.d_model = 256  # reduced from 512
config.griffin.d_rnn = 340  # ~4/3 of d_model
config.griffin.n_layers = 8  # reduced from 12
config.griffin.local_window = 512  # reduced from 1024
config.titans.d_mem = 128  # reduced from 256
config.titans.n_memory_slots = 64  # reduced from 128
config.diffusion.n_diffusion_steps = 500  # reduced for faster training
config.diffusion.denoiser_layers = 4  # reduced from 6
config.diffusion.n_splits_max = 3000
config.decomposition.max_subproblem_size = 1000  # reduced from 1500
config.training.batch_size = 4  # reduced from 8
config.training.learning_rate = 1e-3
config.training.max_steps = 50000  # reduced for Colab runtime limits
config.simulation.pdb_cache_dir = "/content/drive/MyDrive/phylogriffin_data/pdb_cache"
config.simulation.pregen_dir = "/content/drive/MyDrive/phylogriffin_data/pregen_fasta"
# Stage 1 was OOMing due to huge B*N*L token batches.
# Drop the collated batch size so the encoder never sees more than ~1M tokens.
config.training.batch_size = 1
config.training.max_tokens_per_batch = 300_000  # cap single-MSA forward at 300k tokens


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Config: d_model={config.griffin.d_model}, n_layers={config.griffin.n_layers}")

# Create checkpoint directory in Drive
CHECKPOINT_DIR = "/content/drive/MyDrive/phylogriffin_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Using device: cuda
Config: d_model=256, n_layers=8


In [4]:
"""
SETUP: Pre-generate or load ProteinMPNN sequence pools.

This cell runs ONCE. It either:
- Downloads PDB structures and runs ProteinMPNN (first time, ~3-5 hours)
- Loads existing pre-generated data from Drive (subsequent runs, ~1 minute)
"""

import os

from phylogriffin.simulation import (
    build_pregen_index,
    download_representative_pdbs,
    pregenerate_all_backbones,
)

sim_config = config.simulation

# --- Step 1: Clone ProteinMPNN if needed ---
PROTEINMPNN_PATH = "/content/ProteinMPNN"
if not os.path.exists(PROTEINMPNN_PATH):
    print("Cloning ProteinMPNN...")
    !git clone https://github.com/dauparas/ProteinMPNN.git {PROTEINMPNN_PATH}
    !pip install pyvolve

# --- Step 2: Download PDB backbones if needed ---
os.makedirs(sim_config.pdb_cache_dir, exist_ok=True)
existing_pdbs = [f for f in os.listdir(sim_config.pdb_cache_dir) if f.endswith(".pdb")]

if len(existing_pdbs) < sim_config.n_backbones:
    print(f"Downloading {sim_config.n_backbones} PDB structures...")
    pdb_ids = download_representative_pdbs(
        sim_config.pdb_cache_dir,
        n_structures=sim_config.n_backbones,
        resolution_max=sim_config.pdb_resolution_max,
        length_min=sim_config.pdb_length_min,
        length_max=sim_config.pdb_length_max,
    )
    print(f"Downloaded {len(pdb_ids)} PDB structures")
else:
    print(f"Found {len(existing_pdbs)} existing PDB structures")

# --- Step 3: Pre-generate ProteinMPNN sequences if needed ---
os.makedirs(sim_config.pregen_dir, exist_ok=True)

manifest_path = os.path.join(sim_config.pregen_dir, "manifest.json")
if not os.path.exists(manifest_path):
    print(f"Pre-generating ProteinMPNN sequences for all backbones...")
    print(
        f"This will take approximately {sim_config.n_backbones * 0.5:.0f}-{sim_config.n_backbones * 1:.0f} minutes."
    )
    counts = pregenerate_all_backbones(
        sim_config.pdb_cache_dir,
        sim_config.pregen_dir,
        sim_config,
    )
    n_total = sum(counts.values())
    print(f"Pre-generation complete! {n_total} total sequences across {len(counts)} backbones.")
else:
    import json

    with open(manifest_path) as f:
        manifest = json.load(f)
    print(
        f"Loaded existing pre-generated data: {manifest['n_backbones']} backbones, "
        f"{manifest['total_sequences']} total sequences"
    )

# --- Step 4: Build the index for training ---
pregen_index = build_pregen_index(sim_config.pregen_dir)
print(f"Training data index built: {len(pregen_index['files'])} FASTA pools available")

Cloning ProteinMPNN...
Cloning into '/content/ProteinMPNN'...
remote: Enumerating objects: 634, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 634 (delta 137), reused 132 (delta 132), pack-reused 378 (from 1)
Receiving objects: 100% (634/634), 119.90 MiB | 34.03 MiB/s, done.
Resolving deltas: 100% (290/290), done.
RCSB search returned 300 PDB IDs
  6KJ3: chain length 6 outside [50, 500], skipping
  6ANM: chain length 0 outside [50, 500], skipping
Downloaded 8/10 PDB structures...
  6KJ1: chain length 6 outside [50, 500], skipping
  6KJ4: chain length 6 outside [50, 500], skipping
  6KJ2: chain length 6 outside [50, 500], skipping
  6E6O: chain length 47 outside [50, 500], skipping
Downloaded 14/20 PDB structures...
  5XSG: chain length 6 outside [50, 500], skipping
  7TLS: chain length 6 outside [50, 500], skipping
  4REK: chain length 543 outside [50, 500], skipping
  1HJE: chain length 13 outside [50, 500], skipp

In [5]:
# Create the column processor
column_processor = ColumnProcessor(config).to(device)
column_processor.enable_gradient_checkpointing()

# Count parameters
n_params = sum(p.numel() for p in column_processor.parameters())
print(f"Column processor parameters: {n_params:,}")

# Initialize other components (they'll be trained later)
graph_predictor = GraphPredictor(
    d_model=config.griffin.d_model,
    hidden_dims=config.graph.predictor_hidden,
).to(device)

diffusion = DiffusionTreeGenerator(config).to(device)

supertree = SupertreeReconciler(config).to(device)

refinement = RefinementPass(config).to(device)

# Count total parameters
total_params = (
    sum(p.numel() for p in column_processor.parameters())
    + sum(p.numel() for p in graph_predictor.parameters())
    + sum(p.numel() for p in diffusion.parameters())
    + sum(p.numel() for p in supertree.parameters())
    + sum(p.numel() for p in refinement.parameters())
)
print(f"Total parameters (all components): {total_params:,}")

Column processor parameters: 8,508,144
Total parameters (all components): 19,295,677


In [1]:
"""
VERIFY ENVIRONMENT FOR torch.compile
------------------------------------
Check that we have a compatible PyTorch and CUDA setup.
"""

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")

    major, minor = torch.cuda.get_device_capability(0)
    if major < 7:
        print("WARNING: GPU compute capability < 7.0. torch.compile may not work optimally.")
        print("    Consider using mode='default' instead of 'reduce-overhead'.")
        COMPILE_MODE = "default"
    else:
        COMPILE_MODE = "reduce-overhead"
else:
    COMPILE_MODE = "default"
    print("No GPU detected. torch.compile will use CPU backend.")

if not hasattr(torch, "compile"):
    print("WARNING: torch.compile NOT available. Upgrade to PyTorch >= 2.0.")
    print("    pip install torch>=2.0 --upgrade")
    USE_COMPILE = False
else:
    USE_COMPILE = False  # opt-in: set to True to enable torch.compile
    print("torch.compile is available")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Compute capability: (8, 0)
torch.compile is available


In [ ]:
"""
COMPILE MODEL FOR PERFORMANCE
-----------------------------
Apply torch.compile to the ColumnProcessor for training speedup.
The first forward pass after this cell will be SLOW (compilation).
This is normal. Every subsequent pass will use the compiled graph.
"""

import torch

print(f"PyTorch version: {torch.__version__}")
if USE_COMPILE:
    print("Compiling ColumnProcessor...")
    column_processor = torch.compile(column_processor, mode=COMPILE_MODE, dynamic=True)

    print("Running compilation warm-up pass...")
    dummy_msa = torch.randint(0, 20, (2, 50, 200), device=device)
    dummy_mask = dummy_msa != config.pad_idx
    with torch.no_grad():
        _ = column_processor(dummy_msa, dummy_mask)
    print("Done! Subsequent forward passes will be 2-5x faster.")
else:
    print("torch.compile not available (requires PyTorch >= 2.0).")
    print("Continuing without compilation -- training will be slower.")

In [6]:
from phylogriffin.data import MaxTokensCollator, PreGeneratedDataset
from phylogriffin.train.train_column_recon import train_column_reconstruction

# Use MaxTokensCollator to prevent OOM on large MSAs
collator = MaxTokensCollator(config)
dataset = PreGeneratedDataset(
    pregen_dir=sim_config.pregen_dir,
    n_examples_per_epoch=5000,
    n_leaves_range=(50, 500),
    n_sites_range=(200, 1500),
    model=sim_config.pyvolve_model,
    alpha=sim_config.pyvolve_alpha,
    include_trees=False,
    split="train",
)
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    collate_fn=collator,
    pin_memory=True,
)

print(f"Dataset size: {len(dataset)} MSAs")

# Train
STAGE1_PATH = os.path.join(CHECKPOINT_DIR, "stage1_column_recon.pt")

if os.path.exists(STAGE1_PATH):
    print("Loading existing Stage 1 checkpoint...")
    column_processor.load_state_dict(torch.load(STAGE1_PATH, map_location=device))
else:
    print("Starting Stage 1 training...")
    column_processor = train_column_reconstruction(column_processor, dataloader, config, device)
    torch.save(column_processor.state_dict(), STAGE1_PATH)
    print(f"Stage 1 complete! Saved to {STAGE1_PATH}")

Dataset size: 5000 MSAs
Starting Stage 1 training...


Train-1:   0%|          | 0/50000 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.26 GiB. GPU 0 has a total capacity of 79.25 GiB of which 168.81 MiB is free. Including non-PyTorch memory, this process has 79.07 GiB memory in use. Of the allocated memory 78.07 GiB is allocated by PyTorch, and 527.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [7]:
from phylogriffin.data import MaxTokensCollator, PreGeneratedDataset
from phylogriffin.train.train_column_contrast import train_contrastive

collator = MaxTokensCollator(config)

# Create dataloader
contrast_dataset = PreGeneratedDataset(
    pregen_dir=sim_config.pregen_dir,
    n_examples_per_epoch=5000,
    n_leaves_range=(50, 500),
    n_sites_range=(200, 1500),
    model=sim_config.pyvolve_model,
    alpha=sim_config.pyvolve_alpha,
    include_trees=True,
    split="train",
)
contrast_dataloader = DataLoader(
    contrast_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    collate_fn=collator,
    pin_memory=True,
)

print(f"Contrastive dataset size: {len(contrast_dataset)}")

# Freeze early layers
for i in range(min(6, config.griffin.n_layers)):
    for p in column_processor.layers[i].parameters():
        p.requires_grad = False

# Unfreeze Titans memory
if column_processor.titans is not None:
    for p in column_processor.titans.parameters():
        p.requires_grad = True

STAGE2_PATH = os.path.join(CHECKPOINT_DIR, "stage2_contrastive.pt")

if os.path.exists(STAGE2_PATH):
    print("Loading existing Stage 2 checkpoint...")
    column_processor.load_state_dict(torch.load(STAGE2_PATH, map_location=device))
else:
    print("Starting Stage 2 training...")
    column_processor = train_contrastive(column_processor, contrast_dataloader, config, device)
    torch.save(column_processor.state_dict(), STAGE2_PATH)
    print(f"Stage 2 complete! Saved to {STAGE2_PATH}")

Contrastive dataset size: 5000
Starting Stage 2 training...


Train-2:   0%|          | 0/50000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from phylogriffin.data import MaxTokensCollator, PreGeneratedDataset
from phylogriffin.train.train_graph import train_graph_predictor

graph_dataset = PreGeneratedDataset(
    pregen_dir=sim_config.pregen_dir,
    n_examples_per_epoch=5000,
    n_leaves_range=(50, 500),
    n_sites_range=(200, 1500),
    model=sim_config.pyvolve_model,
    alpha=sim_config.pyvolve_alpha,
    include_trees=True,
    split="train",
)
graph_dataloader = DataLoader(
    graph_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    collate_fn=MaxTokensCollator(config),
    pin_memory=True,
)

STAGE3_PATH = os.path.join(CHECKPOINT_DIR, "stage3_graph.pt")

if os.path.exists(STAGE3_PATH):
    print("Loading existing Stage 3 checkpoint...")
    graph_predictor.load_state_dict(torch.load(STAGE3_PATH, map_location=device))
else:
    print("Starting Stage 3 training...")
    # Freeze column processor
    for p in column_processor.parameters():
        p.requires_grad = False

    graph_predictor = train_graph_predictor(
        column_processor, graph_predictor, graph_dataloader, config, device
    )
    torch.save(graph_predictor.state_dict(), STAGE3_PATH)
    print(f"Stage 3 complete! Saved to {STAGE3_PATH}")

In [ ]:
from phylogriffin.data import MaxTokensCollator, PreGeneratedSubproblemDataset
from phylogriffin.train.train_diffusion import train_diffusion

collator = MaxTokensCollator(config)

subproblem_dataset = PreGeneratedSubproblemDataset(
    pregen_dir=sim_config.pregen_dir,
    n_examples_per_epoch=3000,
    n_leaves_range=(100, 1000),
    n_sites_range=(200, 1500),
    model=sim_config.pyvolve_model,
    alpha=sim_config.pyvolve_alpha,
    include_trees=True,
    split="train",
    max_leaves=config.decomposition.max_subproblem_size,
)
subproblem_dataloader = DataLoader(
    subproblem_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    collate_fn=collator,
    pin_memory=True,
)

STAGE4_PATH = os.path.join(CHECKPOINT_DIR, "stage4_diffusion.pt")

if os.path.exists(STAGE4_PATH):
    print("Loading existing Stage 4 checkpoint...")
    diffusion.load_state_dict(torch.load(STAGE4_PATH, map_location=device))
else:
    print("Starting Stage 4 training...")
    # Column processor is frozen
    for p in column_processor.parameters():
        p.requires_grad = False

    diffusion = train_diffusion(column_processor, diffusion, subproblem_dataloader, config, device)
    torch.save(diffusion.state_dict(), STAGE4_PATH)
    print(f"Stage 4 complete! Saved to {STAGE4_PATH}")

In [ ]:
from phylogriffin.data import MaxTokensCollator, PreGeneratedDecomposedDataset
from phylogriffin.train.train_supertree import train_supertree

collator = MaxTokensCollator(config)

decomp_dataset = PreGeneratedDecomposedDataset(
    pregen_dir=sim_config.pregen_dir,
    n_examples_per_epoch=3000,
    n_leaves_range=(500, 5000),
    n_sites_range=(200, 1500),
    model=sim_config.pyvolve_model,
    alpha=sim_config.pyvolve_alpha,
    include_trees=True,
    split="train",
)
decomp_dataloader = DataLoader(
    decomp_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    collate_fn=collator,
    pin_memory=True,
)

STAGE5_PATH = os.path.join(CHECKPOINT_DIR, "stage5_supertree.pt")

if os.path.exists(STAGE5_PATH):
    print("Loading existing Stage 5 checkpoint...")
    supertree.load_state_dict(torch.load(STAGE5_PATH, map_location=device))
else:
    print("Starting Stage 5 training...")
    # Freeze column processor and diffusion
    for p in column_processor.parameters():
        p.requires_grad = False
    for p in diffusion.parameters():
        p.requires_grad = False

    supertree = train_supertree(
        column_processor, diffusion, supertree, decomp_dataloader, config, device
    )
    torch.save(supertree.state_dict(), STAGE5_PATH)
    print(f"Stage 5 complete! Saved to {STAGE5_PATH}")

In [ ]:
from phylogriffin.data import MaxTokensCollator, PreGeneratedErrorTreeDataset
from phylogriffin.train.train_refinement import train_refinement

collator = MaxTokensCollator(config)

error_dataset = PreGeneratedErrorTreeDataset(
    pregen_dir=sim_config.pregen_dir,
    n_examples_per_epoch=2000,
    n_leaves_range=(50, 300),
    n_sites_range=(200, 1000),
    model=sim_config.pyvolve_model,
    alpha=sim_config.pyvolve_alpha,
    include_trees=True,
    split="train",
    d_model=config.griffin.d_model,
    n_nni_swaps=3,
)
error_dataloader = DataLoader(
    error_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    collate_fn=collator,
    pin_memory=True,
)

STAGE6_PATH = os.path.join(CHECKPOINT_DIR, "stage6_refinement.pt")

if os.path.exists(STAGE6_PATH):
    print("Loading existing Stage 6 checkpoint...")
    refinement.load_state_dict(torch.load(STAGE6_PATH, map_location=device))
else:
    print("Starting Stage 6 training...")
    refinement = train_refinement(refinement, error_dataloader, config, device)
    torch.save(refinement.state_dict(), STAGE6_PATH)
    print(f"Stage 6 complete! Saved to {STAGE6_PATH}")

In [ ]:
# Assemble the full PhyloGriffinV3 model with all trained components
model = PhyloGriffinV3(config)
model.column_processor = column_processor
model.graph_predictor = graph_predictor
model.decomposition = HierarchicalDecomposition(config)
model.diffusion = diffusion
model.supertree = supertree
model.refinement = refinement

model = model.to(device)
model.eval()

FULL_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "phylogriffin_v3_full.pt")
torch.save(
    {
        "config": config.to_dict(),
        "model_state_dict": model.state_dict(),
    },
    FULL_MODEL_PATH,
)

print(f"Full model saved to {FULL_MODEL_PATH}")

In [ ]:
# Generate a small test MSA
print("Generating test MSA...")
test_tree = simulate_yule_tree(50, seed=999)
test_msa, test_names = evolve_sequences(test_tree, n_sites=500, model="JTT", alpha=1.0, seed=999)

print(f"Test MSA shape: {test_msa.shape}")
print(f"True tree (first 200 chars): {test_tree[:200]}...")

# Infer tree
print("\nRunning inference...")
inferred_tree = infer_tree(test_msa, test_names, config, model, device=device)

print(f"\nInferred tree (first 200 chars): {inferred_tree[:200]}...")

# Compute RF distance
true_splits = newick_to_splits(test_tree, 50)
inf_splits = newick_to_splits(inferred_tree, 50)
rf = robinson_foulds(true_splits, inf_splits)
print(f"\nRobinson-Foulds distance: {rf:.4f} (0=identical, 1=completely different)")

In [ ]:
# Save a standalone inference script to Drive
INFERENCE_SCRIPT = '''#!/usr/bin/env python3
"""Standalone inference script for PhyloGriffin v3."""
import torch
import sys
from phylogriffin.config import PhyloGriffinConfig
from phylogriffin.inference import PhyloGriffinV3, infer_tree
from phylogriffin.data import load_msa

def main():
    if len(sys.argv) < 3:
        print("Usage: python infer.py <alignment.fa> <model_checkpoint.pt>")
        sys.exit(1)

    msa_path = sys.argv[1]
    checkpoint_path = sys.argv[2]

    # Load MSA
    msa, names = load_msa(msa_path)
    print(f"Loaded MSA: {msa.shape[0]} sequences, {msa.shape[1]} columns")

    # Load config and model
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    config = PhyloGriffinConfig.from_dict(checkpoint['config'])
    model = PhyloGriffinV3(config)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Infer tree
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    tree = infer_tree(msa, names, config, model, device=device)

    # Output
    print(tree)

if __name__ == "__main__":
    main()
'''

with open(os.path.join(CHECKPOINT_DIR, "infer.py"), "w") as f:
    f.write(INFERENCE_SCRIPT)

print("Inference script saved to Drive!")
print(f"To use on your laptop: python {CHECKPOINT_DIR}/infer.py alignment.fa {FULL_MODEL_PATH}")